# RSI-Plateau: Free-T4 Validation (Real Model Path)

**Goal:** Prove the *real* pipeline works on a GPU (Colab/Kaggle free T4) before spending
budget on the Tier-1 matrix. This runs a tiny STaR loop with Qwen2.5-0.5B on a small GSM8K
subset, with the oracle verifier and LoRA SFT for 2 rounds.

**Expected outcome:** non-zero accuracy, round-to-round change, and a `result.json`.
These are **plumbing numbers, not paper numbers** (PRD section 5.9, Tier 0).

**Runtime:** set to GPU (T4). ~10-20 min.

> Kaggle note: replace the first cell with `!git clone <your-repo>` and `%cd`.

In [ ]:
# 1. Environment
!nvidia-smi
!pip install -q -e ".[tier0]"

In [ ]:
# 2. Sanity: torch sees the GPU
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("free VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else 0)

In [ ]:
# 3. Write a GPU smoke config (0.5B, small subset, bf16-safe -> float16 on T4)
import pathlib, yaml

cfg = {
    "run": {"name": "t4_smoke", "seed": 0, "rounds": 2,
            "output_dir": "artifacts/t4_smoke", "use_wandb": False},
    "data": {"dataset": "gsm8k", "train_size": 50, "test_size": 100},
    "model": {"policy": "Qwen/Qwen2.5-0.5B-Instruct", "dtype": "float16",
              "device": "cuda", "judge": None},
    "generation": {"k_samples": 2, "temperature": 0.8, "top_p": 0.95,
                   "max_new_tokens": 256, "batch_size": 8},
    "verifier": {"type": "oracle", "match_acceptance_rate": None, "calibration_size": 0},
    "loop": {"architecture": "star", "frozen_reference_diversity": True},
    "training": {"method": "lora", "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
                 "learning_rate": 0.0001, "epochs": 1, "batch_size": 2, "grad_accum": 4,
                 "max_seq_len": 512},
    "diversity": {"output_entropy": True, "embedding_clustering": False, "self_bleu": True},
    "stats": {"bootstrap_samples": 200, "ci_alpha": 0.05,
              "plateau_abs_delta": 0.005, "plateau_consecutive": 2},
}
pathlib.Path("configs").mkdir(exist_ok=True)
with open("configs/t4_smoke.yaml", "w") as fh:
    yaml.safe_dump(cfg, fh)
print(open("configs/t4_smoke.yaml").read())

In [ ]:
# 4. Run the real loop (HF backend + LoRA SFT) on the T4
!python scripts/run_loop.py --config configs/t4_smoke.yaml 2>&1 | tail -40

In [ ]:
# 5. Inspect the result artifact
import json
d = json.load(open("artifacts/t4_smoke/result.json"))
print("env:", d["metadata"].get("gpu"))
for r in d["result"]["rounds"]:
    print(r["round"], "acc=", round(r["accuracy"], 3),
          "kept=", f"{r['n_train_kept']}/{r['n_train_total']}",
          "self_bleu=", round(r["diversity"].get("self_bleu", -1), 3),
          "train_loss=", r.get("train_loss"))
print("plateau:", d["result"]["plateau"])

## Interpreting the result

Success = the loop ran end-to-end on GPU: sampling worked, oracle filtering kept some
samples, LoRA training completed, evaluation produced a number, and `result.json` exists.

Accuracy will be low (0.5B model, 2 rounds, 50 training questions) — that is expected and
fine. The point is to de-risk the *backends* (HF generate, PEFT/TRL SFT, dtype/device)
before launching the paid Tier-1 matrix.

If this cell block passes, the remaining risk is compute budget, not code.